Быков Владимир Андреевич 465327 J3113


Библиотеки

In [83]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

Ход выполнения работы
1. Выбрал дискретное (Биномиальное) и непрерывное (Нормальное) распределения
2. Сгенерировал выборки по N=1000 для каждого
3. Рассчитал описательные статистики для обеих выборок
4. Построил графики эмпирических и теоретических распределений
5. Исследовал устойчивость характеристик к выбросам

Генерация данных

In [84]:
np.random.seed(107)
# Дискретное распределение
binom_sample = np.random.binomial(n=10, p=0.5, size=1000)

# Непрерывное распределение
norm_sample = np.random.normal(loc=0, scale=1, size=1000)

Фенкция для расчета статистик(половину функций реально переписать руками, зачем?)

ычисляем множество описательных статистик

In [85]:
def calculate_statistics(sample):
    stats_dict = dict()
    
    # Квартили
    stats_dict["Q1"], stats_dict["Q2"], stats_dict["Q3"] = np.quantile(sample, [0.25, 0.5, 0.75])
    
    # Центральные тенденции
    stats_dict["Mean"] = np.mean(sample)
    stats_dict["Median"] = np.median(sample)
    
    mode_result = stats.mode(sample, axis=None, keepdims=True)
    stats_dict["Mode"] = mode_result.mode[0] if mode_result.count[0] > 0 else None  
    
    # Вариабельность разница между наибольшим и наименьшим(разброс)
    stats_dict["Range"] = np.ptp(sample)
    stats_dict["IQR"] = stats_dict["Q3"] - stats_dict["Q1"]
    #lbcgthcbz
    stats_dict["Variance"] = np.var(sample, ddof=1)
    # стандартное отклонение
    stats_dict["Std"] = np.std(sample, ddof=1)
    stats_dict["CV"] = stats_dict["Std"] / stats_dict["Mean"] if stats_dict["Mean"] != 0 else np.nan
    #среднее абсолютное откл
    stats_dict["MAD"] = np.mean(np.abs(sample - stats_dict["Mean"]))
    
    # Форма распределения (оценивает асимметрию и пиковость распределения)
    stats_dict["Skew"] = stats.skew(sample, bias=False)
    stats_dict["Kurtosis"] = stats.kurtosis(sample, fisher=True, bias=False)
    
    # Моменты
    #Показывают форму распределения относительно среднего. Например:
    #2-й момент = дисперсия.
    #3-й и 4-й связаны с асимметрией и эксцессом соответственно.
    
    stats_dict["Raw Moments"] = [np.mean(sample**i) for i in range(1, 6)]
    
    deviations = sample - stats_dict["Mean"]
    stats_dict["Central Moments"] = [np.mean(deviations**i) for i in range(1, 6)]
    
    return stats_dict  

In [86]:
binom_stats = calculate_statistics(binom_sample)
norm_stats = calculate_statistics(norm_sample)

как выглядят binom_stats и norm stats

In [87]:
print("Биномиальное распределение:")
print("")
for k, v in binom_stats.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")
print("")
print("Нормальное распределение:")
print("")
for k, v in norm_stats.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

Биномиальное распределение:

Q1: 4.0000
Q2: 5.0000
Q3: 6.0000
Mean: 5.0420
Median: 5.0000
Mode: 5
Range: 8
IQR: 2.0000
Variance: 2.4667
Std: 1.5706
CV: 0.3115
MAD: 1.2182
Skew: 0.0112
Kurtosis: -0.2073
Raw Moments: [np.float64(5.042), np.float64(27.886), np.float64(165.494), np.float64(1039.942), np.float64(6855.662)]
Central Moments: [np.float64(1.8474111129762604e-16), np.float64(2.464236), np.float64(0.0434321760000014), np.float64(16.928618808912), np.float64(0.8043980726849442)]

Нормальное распределение:

Q1: -0.6984
Q2: -0.0087
Q3: 0.6559
Mean: -0.0121
Median: -0.0087
Mode: -3.6882
Range: 7.5326
IQR: 1.3543
Variance: 1.0463
Std: 1.0229
CV: -84.7856
MAD: 0.8113
Skew: 0.0744
Kurtosis: 0.1879
Raw Moments: [np.float64(-0.012064532884502023), np.float64(1.0454198135024972), np.float64(0.04153877769484537), np.float64(3.472654108808934), np.float64(0.40452037889712367)]
Central Moments: [np.float64(-1.4210854715202004e-17), np.float64(1.0452742605487761), np.float64(0.0793727707923845

графики для биномиального и нормального распределения

In [88]:
fig1 = make_subplots(rows=1, cols=2, subplot_titles=("Эмпирическая и теоретическая PMF", "Функция распределения"))

# Левый график (PMF)
values, counts = np.unique(binom_sample, return_counts=True)
x_binom = np.arange(0, 11)
fig1.add_trace(go.Bar(
    x=values, 
    y=counts, 
    name='Эмпирическая',
    marker=dict(opacity=0.6)
), row=1, col=1)

fig1.add_trace(go.Scatter(
    x=x_binom, 
    y=stats.binom.pmf(x_binom, n=10, p=0.5)*1000,
    mode='lines+markers',
    line=dict(dash='dash', color='red'),
    name='Теоретическая'
), row=1, col=1)

# Правый график (CDF)
x = np.sort(binom_sample)
y = np.arange(1, len(x)+1)/len(x)
fig1.add_trace(go.Scatter(x=x, y=y,mode='lines',line=dict(shape='hvh'),name='ECDF'), row=1, col=2)

fig1.add_trace(go.Scatter(x=x, y=stats.binom.cdf(x, n=10, p=0.5),mode='lines',line=dict(dash='dash', color='red'),name='Теоретическая CDF'), row=1, col=2)

fig1.update_layout(title_text="Биномиальное распределение",showlegend=False,height=400)
fig1.show()

# 2. Графики для нормального распределения
fig2 = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Автоматические бины", "Правило Фридмана-Диакониса", "Ручной выбор (30 бинов)"))

bins_params = ['auto', 'fd', 30]
x_norm = np.linspace(-4, 4, 100)

for i, bins in enumerate(bins_params, 1):
    hist = np.histogram(norm_sample, bins=bins, density=True)
    fig2.add_trace(go.Bar(x=hist[1][:-1],y=hist[0],width=np.diff(hist[1]),name='Эмпирическая',opacity=0.5,showlegend=(i==1)), row=1, col=i)
    fig2.add_trace(go.Scatter(x=x_norm,y=stats.norm.pdf(x_norm),line=dict(color='red'),name='Теоретическая',showlegend=(i==1)), row=1, col=i)

fig2.update_layout(title_text="Нормальное распределение",height=400,bargap=0.01)
fig2.show()

# 3. Boxplot с выбросами
fig3 = go.Figure()
fig3.add_trace(go.Box(
    x=norm_sample,
    name='Распределение',
    boxpoints='outliers',
    marker=dict(
        color='rgb(8,81,156)',
        outliercolor='rgba(219, 64, 82, 0.6)',
        line=dict(outliercolor='rgba(219, 64, 82, 1.0)', outlierwidth=2)
    )
))
fig3.update_layout(title_text="Boxplot с выбросами")
fig3.show()


Эмпирическое распределение очень близко к теоретическому: пики на 5, симметрия, хорошее совпадение.

Небольшие колебания (особенно по краям — 0, 10) допустимы и отражают естественную стохастичность выборки.

Эмпирическая CDF(это функция, которая показывает вероятность того, что случайная величина примет значение меньше или равное заданному.) близка к теоретической, особенно в середине диапазона (4–6).

Лёгкие расхождения на концах объясняются ограниченностью выборки.

Boxplot: Форма симметрична — признак нормального распределения.

Немного выбросов — что нормально для нормального распределения (по правилу 1.5*IQR).

Нет смещения (асимметрии), центр распределения совпадает с медианой.

Анализируем устойчивость к выборам

In [89]:
def add_outliers(sample, perc):
    n_outliers = int(len(sample) * perc)
    # Генерация выбросов
    outliers = np.random.normal(loc=0, scale=5, size=n_outliers)
    return np.concatenate([sample, outliers])


# Постепенное добавление выбросов
metrics = {"Variance": [], "Std": [], "IQR": [], "Range": []}
percentages = np.linspace(0, 0.05, 11)

for p in percentages:
    contaminated = add_outliers(norm_sample, p)
    stats_count = calculate_statistics(contaminated)
    for m in metrics:
        metrics[m].append(stats_count[m])
        
# 4. График изменения метрик
fig4 = go.Figure()
for m, values in metrics.items():
    fig4.add_trace(go.Scatter(
        x=percentages*100,
        y=values,
        mode='lines+markers',
        name=m
    ))

fig4.update_layout(
    title="Изменение мер вариабельности",
    xaxis_title="% выбросов",
    yaxis_title="Значение статистики",
    height=500,
    hovermode="x unified"
)
fig4.show()

Variance std Дисперсия и стандартное отклонение — неустойчивы к выбросам.
Range Размах — крайне неустойчивая мера разброса. Её не стоит использовать, если возможны выбросы.
IQR — устойчивая к выбросам мера разброса. Подходит для анализа данных с потенциальными выбросами.

Заключение
1. Устойчивые статистики: Медиана, IQR и мода показали устойчивость к выбросам.
2. Чувствительные статистики: Среднее, дисперсия и стандартное отклонение значительно меняются при добавлении выбросов.
3. Визуализация: Гистограммы с ручным выбором бинов точнее отражают форму распределения.
4. Практическая значимость: Для данных с потенциальными выбросами рекомендуется использовать устойчивые метрики (медиана, IQR) вместо чувствительных (среднее, дисперсия).
5. Эмпирические распределения: Хорошо согласуются с теоретическими при достаточном объеме данных (N=1000).